<a href="https://colab.research.google.com/github/darkpegassus/AIML-Minor-Deg/blob/main/Hospital_Readmission_Prediction(Case_Study_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *

In [19]:
train_df = pd.read_csv("hospital_readmission_train.csv")
test_df=pd.read_csv("hospital_readmission_test.csv")

In [20]:
train_df.drop_duplicates(inplace=True)

In [21]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Patient_ID             8000 non-null   int64  
 1   age                    8000 non-null   int64  
 2   gender                 8000 non-null   object 
 3   diagnosis_code         8000 non-null   object 
 4   heart_rate_bpm         8000 non-null   int64  
 5   systolic_bp_mmHg       8000 non-null   int64  
 6   diastolic_bp_mmHg      8000 non-null   int64  
 7   glucose_mg_dl          8000 non-null   int64  
 8   oxygen_saturation_pct  8000 non-null   float64
 9   num_procedures         8000 non-null   int64  
 10  days_in_hospital       8000 non-null   int64  
 11  comorbidity_score      8000 non-null   int64  
 12  prior_visits_6m        8000 non-null   int64  
 13  prior_admissions_1y    8000 non-null   int64  
 14  discharge_to           8000 non-null   object 
 15  read

In [22]:
train_df.isnull().sum().sum()

np.int64(0)

In [23]:
train_df.head()

,Patient_ID,age,gender,diagnosis_code,heart_rate_bpm,systolic_bp_mmHg,diastolic_bp_mmHg,glucose_mg_dl,oxygen_saturation_pct,num_procedures,days_in_hospital,comorbidity_score,prior_visits_6m,prior_admissions_1y,discharge_to,readmitted
0,1,60,Male,I25.10,80,111,80,107,96.1,2,1,3,0,0,Skilled Nursing Facility,0
1,2,37,Female,E11.9,84,117,90,191,95.6,2,2,3,1,0,Home,1
2,3,68,Female,E11.9,71,146,69,148,100.0,2,5,2,0,1,Home,0
3,4,66,Male,I25.10,85,126,89,55,96.1,3,2,3,2,0,Rehabilitation Facility,0
4,5,78,Female,I10,90,120,77,163,97.8,0,6,3,1,0,Home,0


In [24]:
categorical_cols = ["gender", "diagnosis_code", "discharge_to"]
train_df_encoded=pd.get_dummies(train_df, columns=categorical_cols, drop_first=True, dummy_na=True, dtype=int)
test_df_encoded=pd.get_dummies(test_df, columns=categorical_cols, drop_first=True, dummy_na=True, dtype=int)

In [25]:
train_df_encoded.head()

,Patient_ID,age,heart_rate_bpm,systolic_bp_mmHg,diastolic_bp_mmHg,glucose_mg_dl,oxygen_saturation_pct,num_procedures,days_in_hospital,comorbidity_score,...,diagnosis_code_I25.10,diagnosis_code_I50.9,diagnosis_code_J18.9,diagnosis_code_J44.9,diagnosis_code_N18.9,diagnosis_code_nan,discharge_to_Home Health Care,discharge_to_Rehabilitation Facility,discharge_to_Skilled Nursing Facility,discharge_to_nan
0,1,60,80,111,80,107,96.1,2,1,3,...,1,0,0,0,0,0,0,0,1,0
1,2,37,84,117,90,191,95.6,2,2,3,...,0,0,0,0,0,0,0,0,0,0
2,3,68,71,146,69,148,100.0,2,5,2,...,0,0,0,0,0,0,0,0,0,0
3,4,66,85,126,89,55,96.1,3,2,3,...,1,0,0,0,0,0,0,1,0,0
4,5,78,90,120,77,163,97.8,0,6,3,...,0,0,0,0,0,0,0,0,0,0


In [26]:
train_df_x=train_df_encoded.drop(['Patient_ID', 'readmitted'], axis=1)
train_df_y=train_df_encoded['readmitted']
test_df_x=test_df_encoded.drop('Patient_ID', axis=1)

In [27]:
x_train, x_val, y_train, y_val = train_test_split(
    train_df_x, train_df_y, test_size=0.2, random_state=42, stratify=train_df_y
)

In [36]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_val_scaled=scaler.transform(x_val)
x_test_scaled=scaler.transform(test_df_x)

In [29]:
model=LogisticRegression(max_iter=1000, penalty="l2", solver="liblinear", random_state=42)
model.fit(x_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [33]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

y_prob = model.predict_proba(x_val_scaled)[:, 1]

for threshold in [0.5, 0.4, 0.3, 0.2]:
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))
    print("Recall:", round(recall_score(y_val, y_pred), 3))
    print("Precision:", round(precision_score(y_val, y_pred, zero_division=0), 3))
    print("F1:", round(f1_score(y_val, y_pred, zero_division=0), 3))


Threshold: 0.5
Confusion matrix:
 [[860 127]
 [389 224]]
Recall: 0.365
Precision: 0.638
F1: 0.465

Threshold: 0.4
Confusion matrix:
 [[657 330]
 [239 374]]
Recall: 0.61
Precision: 0.531
F1: 0.568

Threshold: 0.3
Confusion matrix:
 [[392 595]
 [120 493]]
Recall: 0.804
Precision: 0.453
F1: 0.58

Threshold: 0.2
Confusion matrix:
 [[121 866]
 [ 31 582]]
Recall: 0.949
Precision: 0.402
F1: 0.565


In [34]:
threshold = 0.3
y_prob = model.predict_proba(x_val_scaled)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

In [35]:
accuracy = accuracy_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))
print("Classification Report:")
print(classification_report(y_val, y_pred))

Accuracy: 0.5531
Recall: 0.8042
Precision: 0.4531
F1 Score: 0.5797
Confusion Matrix:
[[392 595]
 [120 493]]
Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.40      0.52       987
           1       0.45      0.80      0.58       613

    accuracy                           0.55      1600
   macro avg       0.61      0.60      0.55      1600
weighted avg       0.65      0.55      0.54      1600



In [37]:
test_prob = model.predict_proba(x_test_scaled)[:, 1]
test_pred = (test_prob >= 0.3).astype(int)

sub_df = pd.DataFrame({
    "Patient_ID": test_df["Patient_ID"],
    "readmitted": test_pred
})

sub_df.to_csv("submission.csv", index=False)

print(sub_df.head())

   Patient_ID  readmitted
0        8001           1
1        8002           1
2        8003           1
3        8004           1
4        8005           1
